#  Clasificación de Anillos en Galaxias — Pipeline Mejorado
## Clases: Sin anillo · Interno · Externo · Interno+Externo



###  Mejoras respecto a versión anterior
- Dataset mapeado a 4 clases operacionales: `none / inner / outer / inner+outer`
- Medición de barra con **3 métodos combinados**: momentos de inercia, ajuste elíptico, gradiente angular
- Extracción de features de anillo: radios normalizados, asimetría azimutal, contraste anular
- Clasificador en cascada **3 niveles** con Random Forest + ExtraTrees ensemble
- Visualizaciones detalladas con perfil radial y árbol de decisión

##  Celda 1 — Imports y Configuración Global

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # Cambiar a 'TkAgg' si usas Jupyter interactivo
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import warnings, os, traceback
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

from astropy.io import fits
from astropy.visualization import make_lupton_rgb
from photutils.background import Background2D, MedianBackground
from astropy.stats import SigmaClip
from skimage.filters import unsharp_mask
from skimage.transform import resize as sk_resize
from scipy.ndimage import gaussian_filter, label as nd_label
from scipy.signal import find_peaks

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier, VotingClassifier)
from sklearn.metrics import (classification_report, confusion_matrix,
                              balanced_accuracy_score, f1_score,
                              ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Mapa de clases (valores originales del dataset) ──────────────────────────
# El dataset usa: 0=sin anillo, 4=interno, 8=externo, 12=int+ext, 2=nuclear, 16=pseudoanillo
# Mapeamos a 4 clases operacionales para el clasificador
RAW_MAP = {0: "sin_anillo", 2: "nuclear", 4: "interno", 8: "externo",
            12: "int+ext", 16: "pseudoanillo"}

# Mapa hacia 4 clases objetivo
# none=0, inner=1, outer=2, inner+outer=3
MERGE_MAP = {0: 0, 2: 0,   # nuclear y sin anillo → none
             4: 1,          # interno → inner
             8: 2,          # externo → outer
             12: 3,         # ambos   → inner+outer
             16: 2}         # pseudoanillo → outer (forma similar)

CLASS_LABELS = {0: "Sin anillo", 1: "Interno", 2: "Externo", 3: "Int+Ext"}
CLASS_COLORS = {0: "#7f8c8d", 1: "#2980b9", 2: "#c0392b", 3: "#d35400"}

PIXEL_SCALE = 0.262   # arcsec/px — SDSS
H0 = 70.0             # km/s/Mpc

print(" Imports OK")
print(f"\nClases objetivo:")
for k,v in CLASS_LABELS.items():
    print(f"  {k}: {v}")

 Imports OK

Clases objetivo:
  0: Sin anillo
  1: Interno
  2: Externo
  3: Int+Ext


##  Celda 2 — Configuración de Rutas

In [2]:
# ══════════════════════════════════════════════════════════════
#  ← EDITA ESTAS LÍNEAS CON TUS RUTAS
# ══════════════════════════════════════════════════════════════
FITS_DIR   = r"../Images_GRZ"
CSV_PATH   = r"../Dataset/dataset.csv"
OUTPUT_DIR = "../output_figuress"
# ══════════════════════════════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Cargar y limpiar CSV ──────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)
print(f"CSV: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Columnas: {list(df_raw.columns)}")

def find_col(df, candidates):
    cl = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cl:
            return cl[c.lower()]
    return None

col_id    = find_col(df_raw, ['objID','objectid','object_id','id'])
col_z     = find_col(df_raw, ['z','redshift','z_spec','zspec'])
col_rings = find_col(df_raw, ['anillos','rings','ring','ring_type',
                               'ring_class','label','clase','class'])
col_ra    = find_col(df_raw, ['ra','RA'])
col_dec   = find_col(df_raw, ['dec','DEC'])

if not all([col_id, col_z, col_rings]):
    missing = [n for n,c in [('objID',col_id),('z',col_z),('anillos',col_rings)] if not c]
    raise ValueError(f"Columnas faltantes: {missing}\nDisponibles: {list(df_raw.columns)}")

df = pd.DataFrame()
df['objID']    = df_raw[col_id].astype(str).str.strip()
df['z']        = pd.to_numeric(df_raw[col_z], errors='coerce').fillna(0.035)
raw_rings      = pd.to_numeric(df_raw[col_rings], errors='coerce').fillna(0).astype(int)
if col_ra:  df['ra']  = pd.to_numeric(df_raw[col_ra],  errors='coerce').fillna(0.0)
if col_dec: df['dec'] = pd.to_numeric(df_raw[col_dec], errors='coerce').fillna(0.0)

# Normalizar valores desconocidos a 0
valid_raw = set(RAW_MAP.keys())
raw_rings = raw_rings.apply(lambda x: x if x in valid_raw else 0)

df['raw_class']  = raw_rings                          # valor original (0,2,4,8,12,16)
df['target']     = raw_rings.map(MERGE_MAP)           # 4 clases: 0,1,2,3
df['label']      = df['target'].map(CLASS_LABELS)

# ── Buscar archivos FITS ──────────────────────────────────────────────────────
def find_fits(obj_id, fdir):
    p = Path(fdir)
    for ext in ['.fits','.fit','.FITS','.FIT']:
        f = p / f"{obj_id}{ext}"
        if f.exists(): return str(f)
    hits = list(p.glob(f"*{str(obj_id)[:16]}*"))
    return str(hits[0]) if hits else None

df['fits_path'] = df['objID'].apply(lambda x: find_fits(x, FITS_DIR))
n_found = df['fits_path'].notna().sum()

print(f"\nFITS encontrados: {n_found:,} / {len(df):,}")
print("\nDistribución de clases (4 clases):")
print("-" * 55)
for k in sorted(CLASS_LABELS):
    n   = (df['target']==k).sum()
    nw  = df[(df['target']==k) & df['fits_path'].notna()].shape[0]
    bar = "█" * int(40 * n / len(df))
    print(f"  {k} | {CLASS_LABELS[k]:<18} | {n:>5} total ({nw:>4} FITS) {bar}")
print("\n Configuración completada")

CSV: 8,528 filas × 5 columnas
Columnas: ['objID', 'ra', 'dec', 'z', 'anillos']

FITS encontrados: 8,053 / 8,528

Distribución de clases (4 clases):
-------------------------------------------------------
  0 | Sin anillo         |  6771 total (6393 FITS) ███████████████████████████████
  1 | Interno            |   857 total ( 806 FITS) ████
  2 | Externo            |   528 total ( 507 FITS) ██
  3 | Int+Ext            |   372 total ( 347 FITS) █

 Configuración completada


In [4]:
from pathlib import Path
from astropy.io import fits

fits_dir = Path(FITS_DIR)

# toma el primer fits/fits.gz que encuentre
fits_files = sorted(list(fits_dir.glob("*.fits")) + list(fits_dir.glob("*.fits.gz")))
print("Encontrados:", len(fits_files))
print("Ejemplo:", fits_files[0] if fits_files else "Ninguno")

with fits.open(fits_files[0]) as hdul:
    data = hdul[0].data

print(data.shape, data.ndim)

Encontrados: 8021
Ejemplo: ..\Images_GRZ\1237648672921485632_grz.fits
(3, 224, 224) 3


##  Celda 3 — Pipeline de Preprocesamiento

In [5]:
# ── Lectura robusta de FITS ───────────────────────────────────────────────────
SINGLE_BAND_MODE = False   # True = FITS de 1 banda (SDSS single-filter stacks)
#con (3, 224, 224) y ndim=3 tu FITS es multibanda (3 bandas).

def read_fits_bands(img_path):
    """Lee FITS y genera 3 bandas sintéticas si es imagen de 1 banda."""
    with fits.open(img_path, memmap=False, mode='readonly') as hdul:
        data = None
        for hdu in hdul:
            if hdu.data is not None and hdu.data.ndim >= 2:
                data = hdu.data.copy()
                break
    if data is None:
        raise ValueError(f"Sin datos en {img_path}")
    data = np.array(data, dtype=np.float32)

    if SINGLE_BAND_MODE or data.ndim == 2:
        while data.ndim > 2:
            data = data[0]
        vmin = np.nanpercentile(data, 1)
        vmax = np.nanpercentile(data, 99)
        img  = np.clip((data - vmin) / (vmax - vmin + 1e-10), 0, 1)
        # Bandas sintéticas: g (suavizada), r (original), z (realzada)
        g_b = (gaussian_filter(img, sigma=0.8) * 0.9 + img * 0.1).astype(np.float32)
        r_b = img.copy().astype(np.float32)
        z_b = np.power(np.clip(gaussian_filter(img, sigma=0.3), 0, None), 0.85).astype(np.float32)
        return g_b, r_b, z_b

    if data.ndim == 3:
        n = data.shape[0]
        return (data[0].astype(np.float32),
                data[min(1,n-1)].astype(np.float32),
                data[min(2,n-1)].astype(np.float32))

    raise ValueError(f"Shape FITS no soportado: {data.shape}")


def subtract_sky(band, box_size=50, filter_size=5):
    band = np.asarray(band, np.float32)
    h, w = band.shape
    bs = max(min(box_size, h // 3, w // 3), 10)
    try:
        bkg = Background2D(band, (bs, bs), filter_size=(filter_size, filter_size),
                           sigma_clip=SigmaClip(sigma=3.0),
                           bkg_estimator=MedianBackground())
        return band - bkg.background
    except Exception:
        return band - np.nanmedian(band)


def apply_unsharp(g, r, z_b, radius=7, amount=2.0):
    return (unsharp_mask(np.asarray(g,  np.float32), radius=radius, amount=amount, preserve_range=False),
            unsharp_mask(np.asarray(r,  np.float32), radius=radius, amount=amount, preserve_range=False),
            unsharp_mask(np.asarray(z_b,np.float32), radius=radius, amount=amount, preserve_range=False))


def adaptive_mask(b1, b2, sigma_thr=2.0, smooth=2.0):
    s   = np.abs(b1 - np.nanmedian(b1)) + np.abs(b2 - np.nanmedian(b2))
    thr = sigma_thr * np.nanstd(s)
    return gaussian_filter((s > thr).astype(float), sigma=smooth) > 0.1


def color_index(b1, b2, clip=(-0.8, 0.8), sigma_thr=2.0, smooth=2.0):
    b1 = np.asarray(b1, np.float32)
    b2 = np.asarray(b2, np.float32)
    n1 = b1 - np.nanmedian(b1)
    n2 = b2 - np.nanmedian(b2)
    ci = np.clip((n1 - n2) / (np.abs(n1) + np.abs(n2) + 1e-10), *clip)
    ci[~adaptive_mask(b1, b2, sigma_thr, smooth)] = np.nan
    return ci


def asinh_norm(data, scale=0.1):
    data = np.asarray(data, np.float32)
    vm   = ~np.isnan(data)
    if not vm.any(): return np.zeros_like(data)
    dv   = data[vm] - data[vm].min()
    sn   = np.arcsinh(dv / scale)
    sn   = (sn - sn.min()) / (sn.max() - sn.min() + 1e-10)
    out  = np.zeros_like(data)
    out[vm] = sn
    return out


def pad_resize(band, target=256):
    h, w   = band.shape
    size   = max(h, w)
    canvas = np.zeros((size, size), np.float32)
    canvas[(size-h)//2:(size-h)//2+h, (size-w)//2:(size-w)//2+w] = band
    return sk_resize(canvas, (target, target), order=1,
                     anti_aliasing=True, preserve_range=True).astype(np.float32)


def process_galaxy(img_path, objID=None):
    """Pipeline completo para una imagen FITS."""
    g_raw, r_raw, z_raw = read_fits_bands(img_path)
    rgb_orig  = make_lupton_rgb(r_raw, g_raw, z_raw, stretch=0.5, Q=8)

    cg = subtract_sky(g_raw); cr = subtract_sky(r_raw); cz = subtract_sky(z_raw)
    rgb_clean = make_lupton_rgb(cr, cg, cz, stretch=0.5, Q=8)

    eg, er, ez = apply_unsharp(cg, cr, cz)
    rgb_enh   = make_lupton_rgb(er, eg, ez, stretch=0.5, Q=8)

    gz = color_index(eg, ez, (-0.8, .8), 2.0, 2.5)
    rz = color_index(er, ez, (-0.6, .6), 2.0, 2.5)
    gr = color_index(eg, er, (-0.5, .5), 2.0, 2.5)

    rgb_ci       = np.zeros((*gz.shape, 3), np.float32)
    rgb_ci[...,0] = asinh_norm(rz, 0.05)
    rgb_ci[...,1] = asinh_norm(gr, 0.05)
    rgb_ci[...,2] = asinh_norm(gz, 0.05)
    gray = np.mean(rgb_ci, axis=2)

    stack = np.stack([pad_resize(asinh_norm(eg)),
                      pad_resize(asinh_norm(er)),
                      pad_resize(asinh_norm(ez))], axis=0)

    return {"objID": objID, "original": rgb_orig, "clean": rgb_clean,
            "enhanced": rgb_enh, "gz_index": gz, "rz_index": rz, "gr_index": gr,
            "rgb_final": rgb_ci, "final_image": gray, "stack_256": stack,
            "bands": (eg, er, ez), "raw_bands": (g_raw, r_raw, z_raw)}


print(f" Pipeline de preprocesamiento listo (SINGLE_BAND_MODE={SINGLE_BAND_MODE})")

 Pipeline de preprocesamiento listo (SINGLE_BAND_MODE=False)


##  Celda 4 — Test Visual de una Imagen

In [6]:
fits_sample = df[df['fits_path'].notna()]['fits_path'].head(1).tolist()
if not fits_sample:
    print("Sin imágenes FITS. Revisa FITS_DIR en la Celda 2.")
else:
    path_test = fits_sample[0]
    print(f"Probando: {path_test}")
    try:
        res = process_galaxy(path_test, objID="test")
        print(f" Pipeline OK: final_image={res['final_image'].shape}")

        fig, axes = plt.subplots(1, 6, figsize=(24, 4))
        fig.patch.set_facecolor('#0d1117')
        panels = [
            (res['original'],                   'Original',       None),
            (res['clean'],                       'BG Sustraído',   None),
            (res['enhanced'],                    'Unsharp',        None),
            (np.nan_to_num(res['gz_index'], 0),  'g-z CI',         'RdBu_r'),
            (res['rgb_final'],                   'RGB Compuesto',  None),
            (res['final_image'],                 'Final (gray)',   'inferno'),
        ]
        for ax, (img, ttl, cm) in zip(axes, panels):
            ax.set_facecolor('#0d1117')
            kw = {'origin': 'lower', 'aspect': 'equal'}
            if cm: kw['cmap'] = cm
            if cm == 'RdBu_r': kw.update({'vmin': -0.5, 'vmax': 0.5})
            ax.imshow(img, **kw)
            ax.set_title(ttl, color='white', fontsize=10)
            ax.axis('off')
        fig.suptitle('Vista previa del pipeline', color='white', fontsize=12)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/test_pipeline.png', dpi=120,
                    bbox_inches='tight', facecolor='#0d1117')
        plt.show()
        print(f" Guardado en {OUTPUT_DIR}/test_pipeline.png")
    except Exception as e:
        traceback.print_exc()
        print(f" Error: {e}")

Probando: ..\Images_GRZ\1237648721210769659.fits
 Pipeline OK: final_image=(224, 224)
 Guardado en ../output_figuress/test_pipeline.png


##  Celda 5 — Extracción de Features

### Features extraídas:
1. **Barra galáctica** (3 métodos): momentos de inercia, ajuste elíptico, gradiente angular
2. **Perfil radial**: picos de brillo, señal de anillo interior/exterior
3. **Índices de color**: gradientes g-z, r-z, g-r por zona anular
4. **Morfología**: concentración, asimetría, elongación
5. **Redshift**: escala física en kpc/px

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
#  MEDICIÓN DE LA BARRA GALÁCTICA
# ─────────────────────────────────────────────────────────────────────────────

def kpc_per_pixel(z, plate_scale=PIXEL_SCALE, H0=H0):
    """Convierte píxeles a kpc dado el redshift."""    
    c_kms = 3e5
    D_Mpc = c_kms * z / (H0 * (1 + z))  # distancia comóvil aproximada
    D_kpc = D_Mpc * 1e3
    rad_per_px = np.radians(plate_scale / 3600.0)
    return D_kpc * rad_per_px


def measure_bar_moments(combined, frac=0.40):
    """
    Método 1: Momentos de inercia de la región central.
    Devuelve longitud, anchura, ángulo y elipticidad de la barra.
    """
    H, W   = combined.shape
    cy, cx = H // 2, W // 2
    r = int(min(cy, cx) * frac)
    reg = combined[cy-r:cy+r, cx-r:cx+r].copy()
    reg = np.clip(reg - np.nanmedian(reg), 0, None)

    yi, xi = np.mgrid[0:reg.shape[0], 0:reg.shape[1]]
    tot    = reg.sum() + 1e-10
    mu_x   = (xi * reg).sum() / tot
    mu_y   = (yi * reg).sum() / tot
    Ixx    = ((xi - mu_x) ** 2 * reg).sum() / tot
    Iyy    = ((yi - mu_y) ** 2 * reg).sum() / tot
    Ixy    = ((xi - mu_x) * (yi - mu_y) * reg).sum() / tot

    evals, evecs = np.linalg.eigh([[Ixx, Ixy], [Ixy, Iyy]])
    lmax, lmin   = max(evals), min(evals)
    bar_px       = 2 * np.sqrt(max(lmax, 0))
    bar_wpx      = 2 * np.sqrt(max(lmin, 0))
    ell          = float(1 - np.sqrt(max(lmin, 0) / (lmax + 1e-10))) if lmax > 0 else 0.0
    angle        = float(0.5 * np.degrees(np.arctan2(2 * Ixy, Ixx - Iyy)))
    return bar_px, bar_wpx, ell, angle


def measure_bar_gradient(combined, n_angles=72):
    """
    Método 2: Gradiente angular — detecta la barra como dirección de máximo
    brillo integrado en sectores angulares de la región central.
    """
    H, W   = combined.shape
    cy, cx = H // 2, W // 2
    r_in   = int(min(cy, cx) * 0.05)
    r_out  = int(min(cy, cx) * 0.35)
    y_idx, x_idx = np.mgrid[0:H, 0:W]
    dist   = np.sqrt((y_idx - cy)**2 + (x_idx - cx)**2)
    theta  = np.arctan2(y_idx - cy, x_idx - cx)

    mask_ann = (dist >= r_in) & (dist <= r_out)
    angles   = np.linspace(-np.pi/2, np.pi/2, n_angles, endpoint=False)
    sector_w = np.pi / n_angles
    profile  = []
    for a in angles:
        diff = np.abs(((theta - a + np.pi) % (2*np.pi)) - np.pi)
        # Considera el sector opuesto también (barra es bidireccional)
        diff_opp = np.abs(((theta - a - np.pi + np.pi) % (2*np.pi)) - np.pi)
        sector_m = mask_ann & ((diff < sector_w) | (diff_opp < sector_w))
        val = combined[sector_m].mean() if sector_m.any() else 0.0
        profile.append(val)

    profile = np.array(profile)
    profile_norm = (profile - profile.min()) / ((profile.max() - profile.min()) + 1e-10)
    bar_idx  = np.argmax(profile)
    bar_ang  = float(np.degrees(angles[bar_idx]))
    contrast = float(profile_norm.max() - profile_norm.mean())
    return bar_ang, contrast, profile_norm


def measure_bar_ellipse(combined, frac=0.45):
    """
    Método 3: Ajuste elíptico a isofota central → elipticidad máxima
    indica la barra.
    """
    H, W   = combined.shape
    cy, cx = H // 2, W // 2
    r_max  = int(min(cy, cx) * frac)
    y_idx, x_idx = np.mgrid[0:H, 0:W]

    max_ell = 0.0
    best_r  = r_max // 2
    radii   = range(int(r_max * 0.15), r_max, max(1, r_max // 12))
    for rv in radii:
        width = max(2, rv // 8)
        ann   = (np.sqrt((y_idx - cy)**2 + (x_idx - cx)**2) >= rv - width) & \
                (np.sqrt((y_idx - cy)**2 + (x_idx - cx)**2) <= rv + width)
        if not ann.any(): continue
        pts_y, pts_x = np.where(ann)
        w            = combined[ann]
        tot          = w.sum() + 1e-10
        mx           = (pts_x * w).sum() / tot
        my           = (pts_y * w).sum() / tot
        Mxx = ((pts_x - mx)**2 * w).sum() / tot
        Myy = ((pts_y - my)**2 * w).sum() / tot
        Mxy = ((pts_x - mx) * (pts_y - my) * w).sum() / tot
        ev, _ = np.linalg.eigh([[Mxx, Mxy], [Mxy, Myy]])
        ell   = float(1 - np.sqrt(max(ev[0], 0) / (ev[1] + 1e-10))) if ev[1] > 0 else 0.0
        if ell > max_ell:
            max_ell = ell
            best_r  = rv
    return float(max_ell), int(best_r)


def extract_bar_features(eg, er, ez_band, z_val):
    """
    Combina los 3 métodos de medición de barra.
    Devuelve diccionario con todas las métricas.
    """
    combined = 0.4 * eg + 0.35 * er + 0.25 * ez_band
    kpc_px   = kpc_per_pixel(z_val)

    # Método 1: Momentos de inercia
    bar_px1, bar_wpx1, ell1, angle1 = measure_bar_moments(combined)
    # Método 2: Gradiente angular
    angle2, bar_contrast, ang_profile = measure_bar_gradient(combined)
    # Método 3: Ajuste elíptico
    ell3, r_ell3 = measure_bar_ellipse(combined)

    # Consenso: promedio ponderado de los métodos
    bar_px   = (bar_px1 + 2 * r_ell3) / 3.0   # promedio ponderado
    bar_kpc  = bar_px * kpc_px

    # Elipticidad consenso
    ell_mean = (ell1 + ell3) / 2.0

    # Ángulo consenso (promedio circular)
    a1r = np.radians(angle1); a2r = np.radians(angle2)
    angle_mean = float(np.degrees(np.arctan2(
        (np.sin(a1r) + np.sin(a2r)) / 2,
        (np.cos(a1r) + np.cos(a2r)) / 2
    )))

    return {
        'bar_length_px':     float(bar_px),
        'bar_length_kpc':    float(bar_kpc),
        'bar_width_px':      float(bar_wpx1),
        'bar_ellipticity':   float(np.clip(ell_mean, 0, 1)),
        'bar_ellip_isophote':float(ell3),
        'bar_contrast':      float(bar_contrast),
        'bar_angle_deg':     float(angle_mean),
        'bar_short':         int(bar_kpc < 5),
        'bar_mid':           int(5 <= bar_kpc < 11),
        'bar_long':          int(bar_kpc >= 11),
        # Ratio longitud/anchura (barra prominente si > 2)
        'bar_ratio':         float(bar_px / (bar_wpx1 + 1e-10)),
        'kpc_per_px':        float(kpc_px),
    }


# ─────────────────────────────────────────────────────────────────────────────
#  PERFIL RADIAL Y SEÑAL DE ANILLO
# ─────────────────────────────────────────────────────────────────────────────

def extract_radial_profile(combined, n_radii=60):
    """
    Perfil de brillo radial + detección de picos (anillos).
    Devuelve el perfil, posiciones de picos y métricas derivadas.
    """
    H, W   = combined.shape
    cy, cx = H // 2, W // 2
    R_max  = min(cy, cx) * 0.95
    radii  = np.linspace(0, R_max, n_radii)

    y_idx, x_idx = np.mgrid[0:H, 0:W]
    dist   = np.sqrt((y_idx - cy)**2 + (x_idx - cx)**2)

    profile = []
    for rv in radii:
        width = max(1.5, rv * 0.12)
        ann   = (dist >= rv - width) & (dist <= rv + width)
        profile.append(combined[ann].mean() if ann.any() else 0.0)

    profile = np.array(profile, dtype=np.float32)
    # Normalizar
    pnorm = (profile - profile.min()) / ((profile.max() - profile.min()) + 1e-10)

    # Detectar picos (señales de anillo)
    # Ignorar núcleo (primeros ~15% del radio)
    nucl_idx = int(n_radii * 0.12)
    peaks, props = find_peaks(pnorm[nucl_idx:], height=0.15, prominence=0.05,
                               distance=max(3, n_radii // 15))
    peaks = peaks + nucl_idx

    # Clasificar picos en inner (20-55% de R_max) y outer (55-95%)
    r_frac  = radii / R_max
    inner_p = [p for p in peaks if 0.20 <= r_frac[p] <= 0.55]
    outer_p = [p for p in peaks if 0.55 <  r_frac[p] <= 0.95]

    inner_sig = float(pnorm[inner_p[0]]) if inner_p else 0.0
    outer_sig = float(pnorm[outer_p[0]]) if outer_p else 0.0
    inner_r   = float(r_frac[inner_p[0]]) if inner_p else 0.0
    outer_r   = float(r_frac[outer_p[0]]) if outer_p else 0.0

    # Ratio entre el pico más brillante y el núcleo
    nuc_mean  = float(pnorm[:nucl_idx].mean())
    ring_nuc_ratio = float((outer_sig + inner_sig) / (nuc_mean + 1e-10))

    return {
        'profile': pnorm, 'radii': radii,
        'inner_ring_sig': inner_sig, 'outer_ring_sig': outer_sig,
        'inner_ring_r':   inner_r,   'outer_ring_r':   outer_r,
        'n_peaks': len(peaks), 'ring_nuc_ratio': ring_nuc_ratio,
        'inner_peaks': inner_p, 'outer_peaks': outer_p,
    }


# ─────────────────────────────────────────────────────────────────────────────
#  FEATURES DE COLOR POR ZONA ANULAR
# ─────────────────────────────────────────────────────────────────────────────

def extract_color_zones(gz_map, rz_map, gr_map, n_zones=5):
    """
    Divide la imagen en anillos concéntricos y calcula índices de color
    por zona. Permite detectar gradientes de color (anillos tienen color
    distinto al disco subyacente).
    """
    H, W   = gz_map.shape
    cy, cx = H // 2, W // 2
    R_max  = min(cy, cx) * 0.9
    y_idx, x_idx = np.mgrid[0:H, 0:W]
    dist   = np.sqrt((y_idx - cy)**2 + (x_idx - cx)**2)
    r_norm = dist / R_max  # 0–1

    feats = {}
    edges = np.linspace(0, 1, n_zones + 1)
    for i in range(n_zones):
        ann = (r_norm >= edges[i]) & (r_norm < edges[i+1])
        for name, cmap in [('gz', gz_map), ('rz', rz_map), ('gr', gr_map)]:
            vals = cmap[ann & ~np.isnan(cmap)]
            if len(vals) > 3:
                feats[f'ci_{name}_z{i}_mean'] = float(vals.mean())
                feats[f'ci_{name}_z{i}_std']  = float(vals.std())
            else:
                feats[f'ci_{name}_z{i}_mean'] = 0.0
                feats[f'ci_{name}_z{i}_std']  = 0.0

    # Gradientes entre zonas (diferencias de color)
    for name in ['gz', 'rz', 'gr']:
        means = [feats[f'ci_{name}_z{i}_mean'] for i in range(n_zones)]
        feats[f'ci_{name}_grad_inner_outer'] = float(
            np.mean(means[3:]) - np.mean(means[:2]))
    return feats


# ─────────────────────────────────────────────────────────────────────────────
#  FUNCIÓN PRINCIPAL: EXTRAE TODOS LOS FEATURES
# ─────────────────────────────────────────────────────────────────────────────

def extract_all_features(proc_result, z_val):
    """Combina todos los features en un único diccionario."""
    eg_f, er_f, ez_f = proc_result['bands']
    combined = 0.4 * eg_f + 0.35 * er_f + 0.25 * ez_f

    # 1. Barra
    bar = extract_bar_features(eg_f, er_f, ez_f, z_val)

    # 2. Perfil radial
    rad = extract_radial_profile(combined)

    # 3. Índices de color por zona
    col_zones = extract_color_zones(
        proc_result['gz_index'], proc_result['rz_index'],
        proc_result['gr_index'], n_zones=5)

    # 4. Morfología global
    gray   = proc_result['final_image']
    H, W   = gray.shape
    cy, cx = H // 2, W // 2
    y_idx, x_idx = np.mgrid[0:H, 0:W]
    dist   = np.sqrt((y_idx - cy)**2 + (x_idx - cx)**2)

    tot    = gray.sum() + 1e-10
    r90    = next((rv for rv in np.linspace(0, min(cy,cx), 100)
                   if gray[dist < rv].sum() / tot > 0.90), float(min(cy,cx)))
    r50    = next((rv for rv in np.linspace(0, min(cy,cx), 100)
                   if gray[dist < rv].sum() / tot > 0.50), float(min(cy,cx) * 0.5))
    conc   = 5 * np.log10((r90 + 1e-3) / (r50 + 1e-3))
    asym   = np.abs(gray - np.rot90(gray, 2)).sum() / tot

    # Elipticidad global de la galaxia
    H_m, W_m = gray.shape
    y_mg, x_mg = np.mgrid[0:H_m, 0:W_m]
    tot2   = gray.sum() + 1e-10
    mx2    = (x_mg * gray).sum() / tot2
    my2    = (y_mg * gray).sum() / tot2
    Mxx2   = ((x_mg - mx2)**2 * gray).sum() / tot2
    Myy2   = ((y_mg - my2)**2 * gray).sum() / tot2
    Mxy2   = ((x_mg - mx2) * (y_mg - my2) * gray).sum() / tot2
    ev2, _ = np.linalg.eigh([[Mxx2, Mxy2], [Mxy2, Myy2]])
    gal_ell = float(1 - np.sqrt(max(ev2[0], 0) / (ev2[1] + 1e-10))) if ev2[1] > 0 else 0.0

    kpc_px = bar['kpc_per_px']

    feats = {
        # — Barra (todos los métodos) —
        **{k: v for k, v in bar.items() if k != 'kpc_per_px'},
        # — Perfil radial —
        'inner_ring_sig':   rad['inner_ring_sig'],
        'outer_ring_sig':   rad['outer_ring_sig'],
        'inner_ring_r':     rad['inner_ring_r'],
        'outer_ring_r':     rad['outer_ring_r'],
        'n_peaks':          float(rad['n_peaks']),
        'ring_nuc_ratio':   rad['ring_nuc_ratio'],
        'has_inner':        float(rad['inner_ring_sig'] > 0.12),
        'has_outer':        float(rad['outer_ring_sig'] > 0.12),
        # — Morfología —
        'concentration':    float(conc),
        'asymmetry':        float(asym),
        'gal_ellipticity':  gal_ell,
        # — Redshift —
        'z':                float(z_val),
        'log_z':            float(np.log10(z_val + 1e-8)),
        'scale_kpc_px':     float(kpc_px),
        # — Features de color por zona —
        **col_zones,
    }
    return feats, bar, rad


FEAT_NAMES_PREVIEW = ['bar_length_kpc','bar_ellipticity','bar_contrast',
                       'inner_ring_sig','outer_ring_sig','ring_nuc_ratio',
                       'concentration','asymmetry','gal_ellipticity','z']
print(f" Extracción de features lista")
print(f"   Features principales: {FEAT_NAMES_PREVIEW}")
print(f"   Total estimado: ~{5 * 3 * 2 + 12 + 9 + 3} features por galaxia")

 Extracción de features lista
   Features principales: ['bar_length_kpc', 'bar_ellipticity', 'bar_contrast', 'inner_ring_sig', 'outer_ring_sig', 'ring_nuc_ratio', 'concentration', 'asymmetry', 'gal_ellipticity', 'z']
   Total estimado: ~54 features por galaxia


## Celda 6 — Procesamiento Batch

In [8]:
def process_batch(df_input, max_per_class=None):
    """
    Procesa todas las galaxias con FITS disponible.
    Devuelve: DataFrame de features, array de labels, lista de objIDs, dict de samples.
    """
    df_v = df_input[df_input['fits_path'].notna()].copy().reset_index(drop=True)
    if len(df_v) == 0:
        print(" No hay fits_path. Revisa FITS_DIR.")
        return pd.DataFrame(), np.array([]), [], {}

    # Muestreo balanceado por clase
    if max_per_class:
        chunks = []
        for cls in sorted(df_v['target'].unique()):
            sub = df_v[df_v['target'] == cls]
            chunks.append(sub.sample(min(len(sub), max_per_class),
                                     random_state=RANDOM_STATE))
        df_v = pd.concat(chunks).reset_index(drop=True)

    feats_all, labels_all, ids_all = [], [], []
    samples = {}
    N    = len(df_v)
    step = max(1, N // 15)
    print(f"Procesando {N} galaxias...")
    print("─" * 65)

    for idx in range(N):
        row = df_v.iloc[idx]
        if idx % step == 0:
            pct = 100 * idx / N
            print(f"  [{idx+1:>5}/{N}] {pct:5.1f}%  "
                  f"{str(row['label']):.<22} id={row['objID']}")
        try:
            lbl    = int(row['target'])
            z_val  = float(row['z'])
            obj_id = str(row['objID'])
            res    = process_galaxy(str(row['fits_path']), objID=obj_id)
            feats_d, bar_d, rad_d = extract_all_features(res, z_val)

            feats_all.append(feats_d)
            labels_all.append(lbl)
            ids_all.append(obj_id)

            # Guardar un sample por clase
            if lbl not in samples:
                samples[lbl] = {'result': res, 'bar_feats': bar_d,
                                 'rad_feats': rad_d, 'label': CLASS_LABELS[lbl],
                                 'feats': feats_d, 'z': z_val}
        except Exception as e:
            if idx < 5:
                import traceback as _tb
                print(f"    ⚠ Error en {row['objID']}: {e}")
                _tb.print_exc()
            elif idx % (step * 5) == 0:
                print(f"    ⚠ Error en {row['objID']}: {e}")

    print("─" * 65)
    print(f" Procesadas: {len(feats_all):,} / {N:,}")

    if not feats_all:
        return pd.DataFrame(), np.array([]), [], {}

    X_out = pd.DataFrame(feats_all).fillna(0.0)
    y_out = np.array(labels_all)

    print("\nDistribución final:")
    for k in sorted(CLASS_LABELS):
        n = (y_out == k).sum()
        print(f"  {CLASS_LABELS[k]:<20}: {n:>5}")
    print(f"  Total features: {X_out.shape[1]}")
    return X_out, y_out, ids_all, samples


# ── Ejecutar ──────────────────────────────────────────────────────────────────
# Ajusta max_per_class según el tamaño de tu dataset y memoria disponible
X_df, y, objIDs, sample_results = process_batch(df, max_per_class=300)

Procesando 1200 galaxias...
─────────────────────────────────────────────────────────────────
  [    1/1200]   0.0%  Sin anillo............ id=1237655464304378007
  [   81/1200]   6.7%  Sin anillo............ id=1237662239091654832
  [  161/1200]  13.3%  Sin anillo............ id=1237664291534340236
  [  241/1200]  20.0%  Sin anillo............ id=1237654605857358005
  [  321/1200]  26.7%  Interno............... id=1237667734489530388
  [  401/1200]  33.3%  Interno............... id=1237667444048658525
  [  481/1200]  40.0%  Interno............... id=1237667108496933522
  [  561/1200]  46.7%  Interno............... id=1237667734518759621
  [  641/1200]  53.3%  Externo............... id=1237656495107211292
  [  721/1200]  60.0%  Externo............... id=1237668293906530358
  [  801/1200]  66.7%  Externo............... id=1237660936091402329
  [  881/1200]  73.3%  Externo............... id=1237657590856482845
  [  961/1200]  80.0%  Int+Ext............... id=1237667733966487677
  [ 1041/

##  Celda 7 — Visualización del Pipeline por Clase

In [11]:
def plot_pipeline_with_bar(res, label="", bar_f=None, rad_f=None, save_path=None):
    """
    9 paneles: Original → BG Sub → Unsharp → 3×CI → RGB → Final + Perfil Radial
    Con barra superpuesta y picos de anillo marcados.
    """
    fig = plt.figure(figsize=(36, 9))
    fig.patch.set_facecolor('#0d1117')
    gs = gridspec.GridSpec(2, 5, figure=fig, hspace=0.35, wspace=0.07)

    # Fila superior: imágenes
    panels_top = [
        (res['original'],                    'Original',       None),
        (res['clean'],                        'BG Sustraído',   None),
        (res['enhanced'],                     'Unsharp',        None),
        (np.nan_to_num(res['gz_index'], 0),   'g-z CI',         'RdBu_r'),
        (res['rgb_final'],                    'RGB Final',      None),
    ]
    for col, (img, ttl, cm) in enumerate(panels_top):
        ax = fig.add_subplot(gs[0, col])
        ax.set_facecolor('#0d1117')
        kw = {'origin': 'lower', 'aspect': 'equal'}
        if cm: kw['cmap'] = cm
        if cm == 'RdBu_r': kw.update({'vmin': -0.5, 'vmax': 0.5})
        ax.imshow(img, **kw)
        ax.set_title(ttl, color='white', fontsize=9, fontweight='bold')
        ax.axis('off')

    # Fila inferior col 0–3: imagen final con barra y anillos superpuestos
    ax_f = fig.add_subplot(gs[1, 0:2])
    ax_f.set_facecolor('#0d1117')
    ax_f.imshow(res['final_image'], origin='lower', aspect='equal', cmap='inferno')
    ax_f.axis('off')
    ax_f.set_title('Final + Barra + Anillos', color='white', fontsize=9, fontweight='bold')

    if bar_f and rad_f:
        H_, W_ = res['final_image'].shape
        cy_, cx_ = H_ // 2, W_ // 2
        # Dibujar barra
        ar  = np.radians(bar_f.get('bar_angle_deg', 0))
        Lh  = bar_f.get('bar_length_px', 30) / 2
        ax_f.plot([cx_ - Lh * np.cos(ar), cx_ + Lh * np.cos(ar)],
                  [cy_ - Lh * np.sin(ar), cy_ + Lh * np.sin(ar)],
                  'r-', lw=3, label=f"Barra ~{bar_f.get('bar_length_kpc',0):.1f}kpc")
        # Dibujar anillos detectados
        R_max = min(cy_, cx_) * 0.95
        th = np.linspace(0, 2 * np.pi, 200)
        if rad_f.get('inner_ring_sig', 0) > 0.12:
            ri = rad_f['inner_ring_r'] * R_max
            ax_f.plot(cx_ + ri * np.cos(th), cy_ + ri * np.sin(th),
                      'c--', lw=1.8, label=f"Inner r={ri:.0f}px")
        if rad_f.get('outer_ring_sig', 0) > 0.12:
            ro = rad_f['outer_ring_r'] * R_max
            ax_f.plot(cx_ + ro * np.cos(th), cy_ + ro * np.sin(th),
                      'orange', ls='--', lw=1.8, label=f"Outer r={ro:.0f}px")
        ax_f.legend(fontsize=7, facecolor='#0d1117', labelcolor='white',
                    loc='lower left', framealpha=0.7)

    # Perfil radial
    ax_r = fig.add_subplot(gs[1, 2:4])
    ax_r.set_facecolor('#1a1a2e')
    if rad_f:
        radii  = rad_f['radii']
        prof   = rad_f['profile']
        R_max  = radii[-1]
        ax_r.plot(radii, prof, color='#ff6b6b', lw=2.5)
        ax_r.fill_between(radii, prof, alpha=0.15, color='#ff6b6b')
        if rad_f.get('inner_ring_sig', 0) > 0.12:
            ri = rad_f['inner_ring_r'] * R_max
            ax_r.axvline(ri, color='cyan', ls='--', lw=2,
                         label=f"Inner r={ri:.0f}px (sig={rad_f['inner_ring_sig']:.2f})")
        if rad_f.get('outer_ring_sig', 0) > 0.12:
            ro = rad_f['outer_ring_r'] * R_max
            ax_r.axvline(ro, color='orange', ls='--', lw=2,
                         label=f"Outer r={ro:.0f}px (sig={rad_f['outer_ring_sig']:.2f})")
        if bar_f:
            Lh = bar_f.get('bar_length_px', 0) / 2
            ax_r.axvline(Lh, color='red', ls=':', lw=2,
                         label=f"½barra ~{bar_f.get('bar_length_kpc',0):.1f}kpc")
        ax_r.set_xlabel('Radio (px)', color='white')
        ax_r.set_ylabel('Brillo norm.', color='white')
        ax_r.set_title('Perfil Radial', color='white', fontweight='bold')
        ax_r.tick_params(colors='white')
        ax_r.legend(fontsize=7.5, facecolor='#0d1117', labelcolor='white')
        for sp in ax_r.spines.values(): sp.set_color('#444')
    else:
        ax_r.text(0.5, 0.5, 'Sin perfil', ha='center', color='white',
                  transform=ax_r.transAxes)

    # Gradiente angular de la barra
    ax_a = fig.add_subplot(gs[1, 4], projection='polar') if False else fig.add_subplot(gs[1, 4])
    ax_a.set_facecolor('#1a1a2e')
    if bar_f:
        # Simular perfil angular
        from scipy.ndimage import gaussian_filter as gf
        eg_f, er_f, ez_f = res['bands']
        combined_b = 0.4 * eg_f + 0.35 * er_f + 0.25 * ez_f
        _, contrast, ang_prof = measure_bar_gradient(combined_b)
        angles_deg = np.linspace(-90, 90, len(ang_prof))
        ax_a.plot(angles_deg, ang_prof, color='#e67e22', lw=2)
        ax_a.fill_between(angles_deg, ang_prof, alpha=0.2, color='#e67e22')
        best_a = bar_f.get('bar_angle_deg', 0)
        ax_a.axvline(best_a, color='red', ls='--', lw=2,
                     label=f"Ángulo={best_a:.1f}°")
        ax_a.set_xlabel('Ángulo (°)', color='white')
        ax_a.set_ylabel('Brillo sector', color='white')
        ax_a.set_title(f'Gradiente Angular\nContraste={contrast:.3f}',
                       color='white', fontweight='bold')
        ax_a.tick_params(colors='white')
        ax_a.legend(fontsize=7, facecolor='#0d1117', labelcolor='white')
        for sp in ax_a.spines.values(): sp.set_color('#444')

    bstr = f" | Barra={bar_f['bar_length_kpc']:.1f}kpc" if bar_f else ""
    fig.suptitle(f"Pipeline — {label}{bstr}", color='white',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    plt.close(fig)


if sample_results:
    print("Visualizando pipeline por clase disponible...")
    for cls in sorted(sample_results):
        s = sample_results[cls]
        print(f"\n  Clase {cls}: {s['label']}")
        plot_pipeline_with_bar(
            s['result'], label=s['label'],
            bar_f=s['bar_feats'], rad_f=s['rad_feats'],
            save_path=f"{OUTPUT_DIR}/pipeline_clase_{cls}.png"
        )
else:
    print("No hay sample_results. Ejecuta la Celda 6 primero.")

Visualizando pipeline por clase disponible...

  Clase 0: Sin anillo

  Clase 1: Interno

  Clase 2: Externo

  Clase 3: Int+Ext


##  Celda 8 — Clasificador en Cascada (4 Clases)

El clasificador usa una estrategia en cascada de 3 niveles:
- **L1**: Sin anillo (0) vs Con anillo (1, 2, 3) — usa morfología global
- **L2**: Solo interno/externo vs Ambos — usa señal relativa inner/outer  
- **L3**: Interno (1) vs Externo (2) — usa barra como feature #1 (corta→interno, larga→externo)

In [12]:
def oversample_jitter(X, y, min_n=150):
    """Oversampling con ruido gaussiano suave para clases minoritarias."""
    Xa, ya = list(X), list(y)
    for cls, cnt in Counter(y).items():
        if cnt < min_n:
            idx  = np.where(y == cls)[0]
            need = min_n - cnt
            ch   = np.random.choice(idx, need, replace=True)
            nz   = np.random.normal(0, 0.004 * (X[ch].std(axis=0) + 1e-8), X[ch].shape)
            Xa.extend(X[ch] + nz)
            ya.extend([cls] * need)
    return np.array(Xa), np.array(ya)


class CascadeRingClassifier4:
    """
    Clasificador en cascada para 4 clases de anillos.
    Clases: 0=Sin anillo, 1=Interno, 2=Externo, 3=Int+Ext
    """
    def __init__(self):
        kw = dict(n_estimators=400, class_weight='balanced',
                  random_state=RANDOM_STATE, n_jobs=-1)
        self.sc1 = RobustScaler(); self.sc2 = RobustScaler(); self.sc3 = RobustScaler()
        # L1: RF + ExtraTrees ensemble
        rf1  = RandomForestClassifier(max_depth=12, **kw)
        et1  = ExtraTreesClassifier(max_depth=12, **kw)
        self.m1 = VotingClassifier([('rf', rf1), ('et', et1)], voting='soft')
        # L2: distingue "solo uno" de "ambos"
        self.m2 = RandomForestClassifier(max_depth=10, **kw)
        # L3: interno vs externo — barra es la feature clave
        rf3  = RandomForestClassifier(max_depth=10, min_samples_leaf=2, **kw)
        et3  = ExtraTreesClassifier(max_depth=10, min_samples_leaf=2, **kw)
        self.m3 = VotingClassifier([('rf', rf3), ('et', et3)], voting='soft')
        self.feat_names = None

    def _bar_first(self, X_df):
        """Reordena: features de barra y anillo al frente."""
        priority = [c for c in X_df.columns if any(k in c for k in
                    ['bar','ring','inner','outer','peak'])]
        others   = [c for c in X_df.columns if c not in priority]
        return X_df[priority + others].values, priority + others

    def fit(self, X_df, y):
        self.feat_names = list(X_df.columns)
        X = X_df.values

        # L1: Sin anillo vs con anillo
        y1 = (y != 0).astype(int)
        Xa1, ya1 = oversample_jitter(X, y1, 200)
        self.m1.fit(self.sc1.fit_transform(Xa1), ya1)
        print(f"  L1 Sin/Con anillo      : {dict(Counter(ya1))}")

        # L2: Solo (inner o outer) vs Ambos (inner+outer)
        m2   = (y != 0)
        y2   = np.where(y[m2] == 3, 1, 0)   # 1 = ambos, 0 = solo uno
        Xa2, ya2 = oversample_jitter(X[m2], y2, 150)
        self.m2.fit(self.sc2.fit_transform(Xa2), ya2)
        print(f"  L2 Solo/Ambos          : {dict(Counter(ya2))}")

        # L3: Interno(1) vs Externo(2) — solo galaxias con UN anillo
        m3   = np.isin(y, [1, 2])
        y3   = np.where(y[m3] == 1, 0, 1)   # 0=interno, 1=externo
        X3_vals, self.feat_order3 = self._bar_first(X_df[m3])
        Xa3, ya3 = oversample_jitter(X3_vals, y3, 120)
        self.m3.fit(self.sc3.fit_transform(Xa3), ya3)
        print(f"  L3 Interno/Externo     : {dict(Counter(ya3))}")
        print(" Cascada entrenada")

    def predict_one(self, feat_series):
        """Predice clase para una galaxia (como pd.Series o dict)."""
        if hasattr(feat_series, 'to_dict'):
            feat_d = feat_series.to_dict()
        else:
            feat_d = feat_series

        X1   = self.sc1.transform([[feat_d[c] for c in self.feat_names]])
        p_ring = float(self.m1.predict_proba(X1)[0][1])
        path = {'p_ring': p_ring}

        if self.m1.predict(X1)[0] == 0:
            path['L1'] = 'sin_anillo'
            return 0, path

        path['L1'] = 'con_anillo'
        X2   = X1.copy()
        p_both = float(self.m2.predict_proba(self.sc2.transform(X2))[0][1])
        path['p_both'] = p_both

        if self.m2.predict(self.sc2.transform(X2))[0] == 1:
            path['L2'] = 'int+ext'
            return 3, path

        path['L2'] = 'solo_uno'
        X3_v = np.array([[feat_d[c] for c in self.feat_order3]])
        p3   = self.m3.predict_proba(self.sc3.transform(X3_v))[0]
        pred3 = int(self.m3.predict(self.sc3.transform(X3_v))[0])
        cls  = 1 if pred3 == 0 else 2
        bk   = feat_d.get('bar_length_kpc', 0)
        path.update({
            'L3':     CLASS_LABELS[cls],
            'p_int':  float(p3[0]),
            'p_ext':  float(p3[1]),
            'bar_kpc': bk,
            'hint':   'corta→interno' if bk < 5 else 'larga→externo' if bk > 11 else 'media',
            'inner_sig': feat_d.get('inner_ring_sig', 0),
            'outer_sig': feat_d.get('outer_ring_sig', 0),
        })
        return cls, path

    def predict(self, X_df):
        return np.array([self.predict_one(row)[0] for _, row in X_df.iterrows()])

    def feat_imp_l3(self):
        """Feature importances del nivel 3 (interno vs externo)."""
        imps = []
        for name, clf in self.m3.named_estimators_.items():
            if hasattr(clf, 'feature_importances_'):
                imps.append(clf.feature_importances_)
        if not imps: return []
        mean_imp = np.mean(imps, axis=0)
        return list(zip(self.feat_order3, mean_imp))

    def feat_imp_l1(self):
        """Feature importances del nivel 1 (sin/con anillo)."""
        imps = []
        for name, clf in self.m1.named_estimators_.items():
            if hasattr(clf, 'feature_importances_'):
                imps.append(clf.feature_importances_)
        if not imps: return []
        return list(zip(self.feat_names, np.mean(imps, axis=0)))


# ── Entrenamiento ─────────────────────────────────────────────────────────────
if len(y) == 0:
    print(" y está vacío. Ejecuta la Celda 6 primero.")
else:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_df, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
    print(f"Train: {len(X_tr)}  Test: {len(X_te)}")
    print("\nEntrenando cascada...")
    cascade = CascadeRingClassifier4()
    cascade.fit(X_tr, y_tr)

    y_pred = cascade.predict(X_te)
    bal    = balanced_accuracy_score(y_te, y_pred)
    f1m    = f1_score(y_te, y_pred, average='macro',    zero_division=0)
    f1w    = f1_score(y_te, y_pred, average='weighted', zero_division=0)

    print(f"\n{'='*50}")
    print(f"  Balanced Accuracy : {bal:.4f}")
    print(f"  F1 Macro          : {f1m:.4f}")
    print(f"  F1 Weighted       : {f1w:.4f}")
    print(f"{'='*50}")
    tnames = [CLASS_LABELS[c] for c in sorted(CLASS_LABELS)]
    print(classification_report(y_te, y_pred, zero_division=0, target_names=tnames))

Train: 960  Test: 240

Entrenando cascada...
  L1 Sin/Con anillo      : {np.int64(1): 720, np.int64(0): 240}
  L2 Solo/Ambos          : {np.int64(0): 480, np.int64(1): 240}
  L3 Interno/Externo     : {np.int64(0): 240, np.int64(1): 240}
 Cascada entrenada

  Balanced Accuracy : 0.3083
  F1 Macro          : 0.2735
  F1 Weighted       : 0.2735
              precision    recall  f1-score   support

  Sin anillo       0.58      0.37      0.45        60
     Interno       0.25      0.47      0.33        60
     Externo       0.26      0.40      0.32        60
     Int+Ext       0.00      0.00      0.00        60

    accuracy                           0.31       240
   macro avg       0.27      0.31      0.27       240
weighted avg       0.27      0.31      0.27       240



## Celda 9 — Resultados y Feature Importance

In [13]:
if 'cascade' not in dir():
    print(" Ejecuta la Celda 8 primero.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.patch.set_facecolor('#0d1117')
    for ax in axes: ax.set_facecolor('#1a1a2e')

    labels_s  = sorted(CLASS_LABELS)
    lbl_names = [CLASS_LABELS[l] for l in labels_s]

    # Matriz de confusión normalizada
    cm_norm = confusion_matrix(y_te, y_pred, labels=labels_s, normalize='true')
    im = axes[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    axes[0].set_xticks(range(len(labels_s)))
    axes[0].set_xticklabels(lbl_names, rotation=30, ha='right', color='white', fontsize=10)
    axes[0].set_yticks(range(len(labels_s)))
    axes[0].set_yticklabels(lbl_names, color='white', fontsize=10)
    for i in range(len(labels_s)):
        for j in range(len(labels_s)):
            axes[0].text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                         fontsize=9, color='white' if cm_norm[i,j] < 0.6 else 'black',
                         fontweight='bold')
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
    axes[0].set_title('Matriz de Confusión\n(normalizada)',
                       color='white', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Predicción', color='white')
    axes[0].set_ylabel('Real', color='white')

    # F1-score por clase
    f1s = f1_score(y_te, y_pred, labels=labels_s, average=None, zero_division=0)
    bars_f1 = axes[1].bar(lbl_names, f1s,
                           color=[CLASS_COLORS[l] for l in labels_s],
                           edgecolor='white', lw=1.5)
    for b, v in zip(bars_f1, f1s):
        axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                     f'{v:.2f}', ha='center', va='bottom',
                     color='white', fontsize=10, fontweight='bold')
    axes[1].set_ylim(0, 1.15)
    axes[1].tick_params(colors='white')
    axes[1].set_title('F1-score por Clase', color='white', fontsize=11, fontweight='bold')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha='right', color='white')
    axes[1].grid(axis='y', alpha=0.2, color='white')
    axes[1].set_ylabel('F1', color='white')

    # Feature importance L3 (Interno vs Externo)
    fi_l3 = cascade.feat_imp_l3()
    if fi_l3:
        fi_l3_s = sorted(fi_l3, key=lambda x: -x[1])[:14]
        fn_r, fv_r = zip(*fi_l3_s)
        fn_r = list(fn_r)[::-1]; fv_r = list(fv_r)[::-1]
        fc_r = ['#c0392b' if 'bar' in n else
                '#2ecc71' if 'ring' in n or 'inner' in n or 'outer' in n else
                '#3498db' for n in fn_r]
        axes[2].barh(range(len(fn_r)), fv_r, color=fc_r, edgecolor='white', lw=0.5)
        axes[2].set_yticks(range(len(fn_r)))
        axes[2].set_yticklabels(fn_r, color='white', fontsize=8.5)
        axes[2].set_xlabel('Importancia', color='white')
        axes[2].tick_params(colors='white')
        axes[2].set_title('Features L3 — Interno vs Externo',
                           color='white', fontsize=11, fontweight='bold')
        axes[2].grid(axis='x', alpha=0.2, color='white')
        axes[2].legend(handles=[
            mpatches.Patch(color='#c0392b', label='Barra'),
            mpatches.Patch(color='#2ecc71', label='Anillo'),
            mpatches.Patch(color='#3498db', label='Otros')],
            facecolor='#1a1a2e', labelcolor='white', fontsize=8)
    else:
        axes[2].text(0.5, 0.5, 'Sin datos L3', ha='center', color='white',
                     transform=axes[2].transAxes)
        axes[2].axis('off')

    fig.suptitle(f'Cascada 4 Clases — Bal.Acc={bal:.3f} | F1 Macro={f1m:.3f}',
                 color='white', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/resultados_4clases.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    plt.close(fig)
    print(f" Guardado en {OUTPUT_DIR}/resultados_4clases.png")

 Guardado en ../output_figuress/resultados_4clases.png


##  Celda 10 — Análisis: Barra vs Tipo de Anillo

In [14]:
if len(y) == 0:
    print(" Ejecuta la Celda 6 primero.")
else:
    ring_m = np.isin(y, [1, 2, 3])
    Xr     = X_df[ring_m]
    yr     = y[ring_m]
    cls3   = [1, 2, 3]
    n3     = [CLASS_LABELS[c] for c in cls3]
    c3     = [CLASS_COLORS[c] for c in cls3]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor('#0d1117')
    for ax in axes.flat: ax.set_facecolor('#1a1a2e')

    # ─ Boxplot: barra por clase ─────────────────────────────────────────────
    db = [Xr[yr == c]['bar_length_kpc'].values for c in cls3]
    bp = axes[0,0].boxplot(db, patch_artist=True, notch=False,
                            medianprops={'color': 'white', 'linewidth': 2.5},
                            flierprops={'marker': '.', 'alpha': 0.3, 'color': 'gray'})
    for p, col in zip(bp['boxes'], c3): p.set_facecolor(col); p.set_alpha(0.85)
    axes[0,0].set_xticks(range(1, 4)); axes[0,0].set_xticklabels(n3, color='white', fontsize=10)
    axes[0,0].set_ylabel('Longitud de Barra (kpc)', color='white')
    axes[0,0].set_title('Barra por Tipo de Anillo', color='white', fontweight='bold')
    axes[0,0].axhline(5,  color='#2980b9', ls='--', alpha=0.7, lw=1.5, label='5kpc')
    axes[0,0].axhline(11, color='#c0392b', ls='--', alpha=0.7, lw=1.5, label='11kpc')
    axes[0,0].legend(fontsize=8, facecolor='#0d1117', labelcolor='white')
    axes[0,0].grid(axis='y', alpha=0.2, color='white')
    axes[0,0].tick_params(colors='white')

    # ─ Violin: barra ────────────────────────────────────────────────────────
    vp = axes[0,1].violinplot(db, positions=range(1, 4), showmedians=True, showextrema=True)
    for body, col in zip(vp['bodies'], c3): body.set_facecolor(col); body.set_alpha(0.7)
    axes[0,1].set_xticks(range(1, 4)); axes[0,1].set_xticklabels(n3, color='white')
    axes[0,1].set_ylabel('Longitud (kpc)', color='white')
    axes[0,1].set_title('Distribución (Violin)', color='white', fontweight='bold')
    axes[0,1].grid(axis='y', alpha=0.2, color='white'); axes[0,1].tick_params(colors='white')

    # ─ Barra vs señal de anillos ─────────────────────────────────────────────
    for cls, col, nm in zip(cls3, c3, n3):
        m = yr == cls
        axes[0,2].scatter(Xr[m]['bar_length_kpc'],
                          Xr[m]['inner_ring_sig'] + Xr[m]['outer_ring_sig'],
                          c=col, label=nm, alpha=0.6, s=40, edgecolors='none')
    axes[0,2].axvline(5,  color='#2980b9', ls='--', alpha=0.7, lw=1.5)
    axes[0,2].axvline(11, color='#c0392b', ls='--', alpha=0.7, lw=1.5)
    axes[0,2].set_xlabel('Longitud Barra (kpc)', color='white')
    axes[0,2].set_ylabel('Señal total de anillo', color='white')
    axes[0,2].set_title('Barra vs Señal de Anillo', color='white', fontweight='bold')
    axes[0,2].legend(fontsize=9, facecolor='#0d1117', labelcolor='white')
    axes[0,2].tick_params(colors='white'); axes[0,2].grid(alpha=0.2, color='white')

    # ─ Inner vs Outer ring signal ────────────────────────────────────────────
    for cls, col, nm in zip(cls3, c3, n3):
        m = yr == cls
        axes[1,0].scatter(Xr[m]['inner_ring_sig'], Xr[m]['outer_ring_sig'],
                          c=col, label=nm, alpha=0.6, s=40, edgecolors='none')
    axes[1,0].set_xlabel('Señal Anillo Interno', color='white')
    axes[1,0].set_ylabel('Señal Anillo Externo', color='white')
    axes[1,0].set_title('Inner sig vs Outer sig', color='white', fontweight='bold')
    axes[1,0].legend(fontsize=9, facecolor='#0d1117', labelcolor='white')
    axes[1,0].tick_params(colors='white'); axes[1,0].grid(alpha=0.2, color='white')

    # ─ Elipticidad barra vs tipo ──────────────────────────────────────────────
    de = [Xr[yr == c]['bar_ellipticity'].values for c in cls3]
    bp2 = axes[1,1].boxplot(de, patch_artist=True, notch=False,
                             medianprops={'color': 'white', 'linewidth': 2.5})
    for p, col in zip(bp2['boxes'], c3): p.set_facecolor(col); p.set_alpha(0.85)
    axes[1,1].set_xticks(range(1, 4)); axes[1,1].set_xticklabels(n3, color='white', fontsize=10)
    axes[1,1].set_ylabel('Elipticidad de Barra', color='white')
    axes[1,1].set_title('Elipticidad vs Tipo de Anillo', color='white', fontweight='bold')
    axes[1,1].grid(axis='y', alpha=0.2, color='white'); axes[1,1].tick_params(colors='white')

    # ─ Histograma de N° de picos ──────────────────────────────────────────────
    for cls, col, nm in zip(cls3, c3, n3):
        m = yr == cls
        counts = Counter(Xr[m]['n_peaks'].astype(int).values)
        xs = sorted(counts.keys())
        axes[1,2].bar([x + 0.25*(cls-1) for x in xs],
                      [counts[x] for x in xs],
                      width=0.25, color=col, alpha=0.85, label=nm)
    axes[1,2].set_xlabel('N° de Picos (perfil radial)', color='white')
    axes[1,2].set_ylabel('Galaxias', color='white')
    axes[1,2].set_title('Número de Picos por Clase', color='white', fontweight='bold')
    axes[1,2].legend(fontsize=9, facecolor='#0d1117', labelcolor='white')
    axes[1,2].tick_params(colors='white'); axes[1,2].grid(axis='y', alpha=0.2, color='white')

    fig.suptitle('Análisis Físico: Barra + Señal de Anillo → Clasificación',
                 color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/barra_anillo_4clases.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    plt.close(fig)

    print("\nEstadísticas:")
    print(f"{'Clase':<22} {'Barra(kpc)':>12} {'Std':>8} {'Inner sig':>10} {'Outer sig':>10}")
    print("─" * 65)
    for cls in cls3:
        mask = yr == cls
        bk   = Xr[mask]['bar_length_kpc']
        ins  = Xr[mask]['inner_ring_sig']
        outs = Xr[mask]['outer_ring_sig']
        print(f"  {CLASS_LABELS[cls]:<20} {bk.mean():>8.2f} ± {bk.std():>5.2f}  "
              f"{ins.mean():>8.3f}    {outs.mean():>8.3f}")


Estadísticas:
Clase                    Barra(kpc)      Std  Inner sig  Outer sig
─────────────────────────────────────────────────────────────────
  Interno                  5.38 ±  2.71     0.030       0.000
  Externo                  5.87 ±  2.90     0.031       0.000
  Int+Ext                  5.45 ±  2.41     0.035       0.000


##  Celda 11 — Cascada Paso a Paso

In [15]:
def plot_cascade_demo(res, feat_d, true_cls, cascade_m, objID=None, save_path=None):
    """Visualiza la cascada de decisión paso a paso para una galaxia."""
    if isinstance(feat_d, pd.Series):
        feat_dict = feat_d.to_dict()
    else:
        feat_dict = feat_d
    pred, path = cascade_m.predict_one(feat_dict)
    correct    = (pred == true_cls)

    fig = plt.figure(figsize=(26, 10))
    fig.patch.set_facecolor('#0d1117')
    gs  = gridspec.GridSpec(2, 5, figure=fig, hspace=0.4, wspace=0.3)

    # Fila 0: imágenes del pipeline
    for col, (img, ttl, cm) in enumerate(zip(
        [res['original'], res['enhanced'],
         np.nan_to_num(res['gz_index'], nan=0), res['rgb_final'], res['final_image']],
        ['Original', 'Enhanced', 'g-z Color', 'RGB', 'Final'],
        [None, None, 'RdBu_r', None, 'inferno']
    )):
        ax = fig.add_subplot(gs[0, col]); ax.set_facecolor('#0d1117')
        kw = {'origin': 'lower', 'aspect': 'equal'}
        if cm: kw['cmap'] = cm
        if cm == 'RdBu_r': kw.update({'vmin': -0.5, 'vmax': 0.5})
        ax.imshow(img, **kw)
        ax.set_title(ttl, color='white', fontsize=9, fontweight='bold')
        ax.axis('off')

    # Fila 1, col 0-1: árbol de decisión textual
    ax_d = fig.add_subplot(gs[1, :2]); ax_d.set_facecolor('#0d1117'); ax_d.axis('off')
    ax_d.set_xlim(0, 1); ax_d.set_ylim(0, 1)
    ax_d.text(0.5, 0.97, 'Árbol de Decisión', ha='center', color='white',
              fontsize=11, fontweight='bold', transform=ax_d.transAxes)
    steps = []
    p_ring_str = f"{path.get('p_ring',0)*100:.1f}%"
    steps.append(f"► L1: {'Con anillo' if path.get('L1')=='con_anillo' else 'Sin anillo'}  "
                 f"(p_ring={p_ring_str})")
    if 'L2' in path:
        p_both_str = f"{path.get('p_both',0)*100:.1f}%"
        steps.append(f"► L2: {path['L2']}  (p_ambos={p_both_str})")
    if 'L3' in path:
        bk   = path.get('bar_kpc', 0)
        hint = path.get('hint', '')
        steps.append(f"► L3: {path['L3']}  (barra={bk:.1f}kpc → {hint})")
        steps.append(f"     p_int={path.get('p_int',0):.3f}  "
                     f"p_ext={path.get('p_ext',0):.3f}")
        steps.append(f"     inner_sig={path.get('inner_sig',0):.3f}  "
                     f"outer_sig={path.get('outer_sig',0):.3f}")
    for i, s in enumerate(steps):
        ax_d.text(0.04, 0.80 - i * 0.14, s, color='#f0f0f0', fontsize=8.5,
                  transform=ax_d.transAxes, fontfamily='monospace')
    for y_r, cls, lbl in [(0.22, true_cls, f"Real: {CLASS_LABELS[true_cls]}"),
                           (0.08, pred,     f"Pred: {CLASS_LABELS[pred]}")]:
        ax_d.add_patch(plt.Rectangle((0.05, y_r), 0.9, 0.10,
                                      color=CLASS_COLORS.get(cls, 'gray'),
                                      alpha=0.85, transform=ax_d.transAxes))
        ax_d.text(0.5, y_r + 0.05, lbl, ha='center', va='center', color='white',
                  fontsize=9, fontweight='bold', transform=ax_d.transAxes)
    res_str = "✓ CORRECTO" if correct else "✗ ERROR"
    ax_d.text(0.5, 0.01, res_str, ha='center',
              color='#2ecc71' if correct else '#e74c3c',
              fontsize=13, fontweight='bold', transform=ax_d.transAxes)

    # Perfil radial
    ax_r = fig.add_subplot(gs[1, 2]); ax_r.set_facecolor('#1a1a2e')
    eg_f, er_f, ez_f = res['bands']
    comb = 0.4 * eg_f + 0.35 * er_f + 0.25 * ez_f
    rad_f = extract_radial_profile(comb)
    radii = rad_f['radii']; prof = rad_f['profile']
    R_max = radii[-1]
    ax_r.plot(radii, prof, color='#ff6b6b', lw=2)
    ax_r.fill_between(radii, prof, alpha=0.15, color='#ff6b6b')
    if rad_f['inner_ring_sig'] > 0.12:
        ax_r.axvline(rad_f['inner_ring_r'] * R_max, color='cyan', ls='--', lw=2, label='inner')
    if rad_f['outer_ring_sig'] > 0.12:
        ax_r.axvline(rad_f['outer_ring_r'] * R_max, color='orange', ls='--', lw=2, label='outer')
    bk_px = feat_dict.get('bar_length_px', feat_dict.get('bar_length_kpc', 0))
    ax_r.axvline(bk_px / 2, color='red', ls=':', lw=2, label='½barra')
    ax_r.set_xlabel('Radio (px)', color='white'); ax_r.set_ylabel('Brillo', color='white')
    ax_r.set_title('Perfil Radial', color='white', fontweight='bold')
    ax_r.tick_params(colors='white')
    ax_r.legend(fontsize=7, facecolor='#0d1117', labelcolor='white')
    for sp in ax_r.spines.values(): sp.set_color('#444')

    # Probabilidades L3 o barra L1
    ax_p = fig.add_subplot(gs[1, 3:]); ax_p.set_facecolor('#1a1a2e')
    if 'p_int' in path:
        probs = [path['p_int'], path['p_ext']]
        plbs  = [f"Interno\n(barra<5kpc)", f"Externo\n(barra>11kpc)"]
        bpp   = ax_p.bar(plbs, probs, color=[CLASS_COLORS[1], CLASS_COLORS[2]],
                          edgecolor='white', lw=1.5)
        for b, v in zip(bpp, probs):
            ax_p.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                      f'{v:.3f}', ha='center', va='bottom',
                      color='white', fontsize=11, fontweight='bold')
        ax_p.set_ylim(0, 1.2)
        ax_p.set_title(f"L3 — {path.get('bar_kpc',0):.1f}kpc ({path.get('hint','')})",
                        color='white', fontweight='bold')
    else:
        ax_p.text(0.5, 0.5, f"Clase: {CLASS_LABELS[pred]}\np_ring={path.get('p_ring',0):.3f}",
                  ha='center', va='center', color='white', fontsize=12,
                  transform=ax_p.transAxes)
        ax_p.axis('off')
    ax_p.tick_params(colors='white')
    ax_p.set_ylabel('Probabilidad', color='white')
    for sp in ax_p.spines.values(): sp.set_color('#444')

    tc = '#2ecc71' if correct else '#e74c3c'
    fig.suptitle(
        f"Demo Cascada | Real: {CLASS_LABELS[true_cls]} → Pred: {CLASS_LABELS[pred]}  {res_str}",
        color=tc, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=110, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    plt.close(fig)
    return pred, path


if 'cascade' not in dir():
    print(" Ejecuta la Celda 8 primero.")
elif len(X_te) == 0:
    print(" Sin datos de test.")
else:
    print("Visualizando un ejemplo por clase en el test set...")
    seen = set()
    for idx in range(len(X_te)):
        cls_t = int(y_te[idx])
        if cls_t in seen: continue
        feat_s = X_te.iloc[idx]
        s = sample_results.get(cls_t, list(sample_results.values())[0])
        print(f"\n  Clase {cls_t}: {CLASS_LABELS.get(cls_t,'?')}")
        plot_cascade_demo(s['result'], feat_s, cls_t, cascade,
                          save_path=f"{OUTPUT_DIR}/cascada_clase_{cls_t}.png")
        seen.add(cls_t)
        if len(seen) == len(CLASS_LABELS): break

Visualizando un ejemplo por clase en el test set...

  Clase 0: Sin anillo

  Clase 2: Externo

  Clase 3: Int+Ext

  Clase 1: Interno


##  Celda 12 — Validación Cruzada Estratificada

In [16]:
if len(y) == 0:
    print(" Ejecuta la Celda 6 primero.")
else:
    print("Ejecutando 5-fold cross-validation...")
    skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    bal_scores = []; f1_scores = []

    for fold, (tr_i, te_i) in enumerate(skf.split(X_df, y)):
        Xtr_f, Xte_f = X_df.iloc[tr_i], X_df.iloc[te_i]
        ytr_f, yte_f = y[tr_i], y[te_i]

        c_f = CascadeRingClassifier4()
        # Silenciar output del entrenamiento en CV
        import io, contextlib
        with contextlib.redirect_stdout(io.StringIO()):
            c_f.fit(Xtr_f, ytr_f)
        yp_f = c_f.predict(Xte_f)

        ba = balanced_accuracy_score(yte_f, yp_f)
        f1 = f1_score(yte_f, yp_f, average='macro', zero_division=0)
        bal_scores.append(ba); f1_scores.append(f1)
        print(f"  Fold {fold+1}/5 — Bal.Acc={ba:.4f}  F1 Macro={f1:.4f}")

    print(f"\n{'═'*45}")
    print(f"  Bal.Acc = {np.mean(bal_scores):.4f} ± {np.std(bal_scores):.4f}")
    print(f"  F1 Macro = {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
    print(f"{'═'*45}")

    # Gráfica de resultados por fold
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor('#0d1117')
    for ax in axes: ax.set_facecolor('#1a1a2e')

    folds = list(range(1, 6))
    for ax, scores, name, col in zip(
        axes,
        [bal_scores, f1_scores],
        ['Balanced Accuracy', 'F1 Macro'],
        ['#3498db', '#e67e22']
    ):
        ax.bar(folds, scores, color=col, edgecolor='white', lw=1.5, alpha=0.85)
        ax.axhline(np.mean(scores), color='white', ls='--', lw=2,
                   label=f'Media={np.mean(scores):.3f}')
        ax.fill_between([0.5, 5.5],
                         [np.mean(scores) - np.std(scores)] * 2,
                         [np.mean(scores) + np.std(scores)] * 2,
                         color='white', alpha=0.08)
        ax.set_xticks(folds); ax.set_xticklabels([f'Fold {i}' for i in folds], color='white')
        ax.set_ylabel(name, color='white'); ax.set_ylim(0, 1.05)
        ax.tick_params(colors='white'); ax.set_title(name, color='white', fontweight='bold')
        ax.legend(fontsize=9, facecolor='#0d1117', labelcolor='white')
        ax.grid(axis='y', alpha=0.2, color='white')
        for sp in ax.spines.values(): sp.set_color('#444')
        for i, v in enumerate(scores):
            ax.text(i+1, v + 0.01, f'{v:.3f}', ha='center', va='bottom',
                    color='white', fontsize=9, fontweight='bold')

    fig.suptitle('Validación Cruzada 5-Fold', color='white', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/cross_validation.png', dpi=130,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show(); plt.close(fig)
    print(f" Guardado en {OUTPUT_DIR}/cross_validation.png")

Ejecutando 5-fold cross-validation...
  Fold 1/5 — Bal.Acc=0.3167  F1 Macro=0.2801
  Fold 2/5 — Bal.Acc=0.3167  F1 Macro=0.2807
  Fold 3/5 — Bal.Acc=0.3417  F1 Macro=0.3021
  Fold 4/5 — Bal.Acc=0.3208  F1 Macro=0.2776
  Fold 5/5 — Bal.Acc=0.3542  F1 Macro=0.3045

═════════════════════════════════════════════
  Bal.Acc = 0.3300 ± 0.0152
  F1 Macro = 0.2890 ± 0.0118
═════════════════════════════════════════════
 Guardado en ../output_figuress/cross_validation.png
